In [ ]:
# Step 0 (Audio Front-end) -- Silero VAD / GTCRN / MVDR-DAS beamforming
# Enable a GPU accelerator in this Kaggle notebook's settings for faster VAD/GTCRN runs.
!nvidia-smi || echo 'no GPU visible -- CPU-only run, still fine'

In [ ]:
!rm -rf OneVoice
!git clone https://github.com/Khanhhh239/OneVoice.git
%cd OneVoice
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Add your VI/EN/ZH/KO clean speech + noise clips before running the tests.
# Easiest on Kaggle: attach a Dataset via "+ Add Input", then copy files in, e.g.:
#
# import shutil, glob
# for lang in ['vi', 'en', 'zh', 'ko']:
#     for f in glob.glob(f'/kaggle/input/onevoice-audio/{lang}/*.wav'):
#         shutil.copy(f, f'data/clean/{lang}/')
# for f in glob.glob('/kaggle/input/onevoice-audio/noise/*.wav'):
#     shutil.copy(f, 'data/noise/')

import os
for lang in ['vi', 'en', 'zh', 'ko']:
    n = len([f for f in os.listdir(f'data/clean/{lang}') if not f.startswith('.')])
    print(f'data/clean/{lang}: {n} file(s)')
n_noise = len([f for f in os.listdir('data/noise') if not f.startswith('.')])
print(f'data/noise: {n_noise} file(s) (0 = synthetic factory-noise fallback will be used)')

In [ ]:
%cd src
!python mix_noise.py

In [ ]:
!python test_vad.py

In [ ]:
!python test_denoise.py

In [ ]:
!python test_beamform.py

In [ ]:
import pandas as pd
for name in ['vad_results', 'denoise_results', 'beamform_results']:
    path = f'../outputs/{name}.csv'
    if os.path.exists(path):
        print(f'\n=== {name} ===')
        display(pd.read_csv(path))
    else:
        print(f'{name}.csv not found (no matching input data?)')